### Practice: Large Language Models and Their Implications
<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

In this notebook, you're gonna play with some of the largest language models on the Internet.

_Based on works of: Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin,  Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih, Maria Ivanova.

### Part 1: prompt engineering (4 points total)

Your task is to prompt-engineer a **base** language model into solving a few tasks for you. A base model only *continues text*, it was never trained to help you, so you steer it with careful prompts and few-shot examples.

__Which model?__ run a base model locally, for example [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base).

__Deliverable:__ for each of the 4 problems, show your **prompt(s)**, the **code** you ran, the **model's output**, and a short **description** of your solution.

__How a base model behaves:__ it continues whatever you write. Begin a dialogue and it continues the dialogue;
supply a few worked examples and it imitates their format. Load the model in the two cells below, then use the
`generate()` helper in your solutions.

__It is fine to re-run and tweak.__ Base models are sensitive to wording: if the output is off, adjust the prompt
or add a few-shot example. If you need many retries for one task, your prompt probably needs rethinking.

In [1]:
# %pip install -q "transformers>=5.0" "accelerate>=1.0"
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm, trange

device = 'cuda'
major, _ = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (compute 7.5) has no native bf16 -> use fp16
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(), "| dtype:", dtype)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()


def generate(prompt, max_new_tokens=64, **kwargs):
    """Continue `prompt` with the base model; return only the newly generated text.
    Pass do_sample=True (+ temperature/top_p/top_k) for creative output, or nothing for greedy."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# smoke test: a base model just continues the text
print(generate("Once upon a time, deep in the forest,", max_new_tokens=40,
               do_sample=True, temperature=0.8, top_p=0.9))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

 there lived a dragon. The dragon had no eyes. A lot of the forest animals were afraid of it. The dragon was sad. One day, a little girl found the dragon. The little girl


__Task 1 (1 pt):__ arange a conversation between any two of the following:

- a celebrity or politician of your choice
- any fictional character (except Darth Vader)
- yourself

Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

In [ ]:
prompt_with_names_only = """
Arange a conversation (5-10 replics) between Jason Statham and Emperor Palpatin.
"""
print(generate(prompt_with_names_only, max_new_tokens=200,
               do_sample=True, temperature=0.8, top_p=0.9))

Certainly! Here's an imagined conversation between Jason Statham and Emperor Palpatine, blending elements from their respective fictional worlds:

---

**Palpatine:** Jason, I must admit, I have not encountered a man like you in quite some time. Your reputation precedes you.

**Jason Statham:** Emperor Palpatine, if I may, you’re not the first to make that claim. But let me assure you, I prefer my enemies dead rather than talking.

**Palpatine:** Dead? My dear Jason, such language is beneath us both. We are in a realm where diplomacy and strategy reign supreme. Perhaps we can discuss your motivations for being here?

**Jason Statham:** Motivations? I came here to settle a score, and I intend to settle it. There are some debts I need to collect.

**Palpatine:** Ah, debts. A common thread among us all. But I assure you, Jason, I have no intention of causing you harm.


In [ ]:
prompt_with_additional_information = """
Arange a conversation (5-10 replics) between Jason Statham (English action movie actor) and Emperor Palpatin (sith lord and ruler of Galactic Empire from Star Wars).
"""
print(generate(prompty_with_names_only, max_new_tokens=200,
               do_sample=True, temperature=0.8, top_p=0.9))

**Jason Statham:** *[adjusting his trench coat, taking a sip from his coffee]* "Palpatin, I've heard the rumors about your latest scheme. Are you really planning to conquer the galaxy with that Force thing?"

**Emperor Palpatin:** *[smiling, leaning back in his throne]* "Ah, Jason. You're quite sharp, aren't you? The Force is merely a tool, a means to an end. My goal is to unite the galaxy under the might of the Empire."

**Jason Statham:** *[crossing his arms, eyeing the Emperor]* "Unite it? Under your rule? Sounds like a bad idea to me. What's in it for the rest of the galaxy? Just more tyranny and oppression?"

**Emperor Palpatin:** *[leaning forward, his voice low]* "Tyranny? Perhaps. But order. Stability. The galaxy has been plagued by chaos, Jason. I offer it a chance at peace."




**Описание решения**: я вписал в промт инструкцию из задания (arange conversation between...) и добавил рекомендацию про объём ответа (диалог из 5-10 реплик). Для генерации запустил функцию generate с параметрами do_sample=True, temperature=0.8 и top_p=0.9.

**Вывод**: при добавлении дополнительных сведений о действующих лицах в промт Qwen предпочитает использовать в диалоге много информации из контекста, и содержание диалога становится более конкретным. Плюс добавились ремарки о действиях героев во время диалога.

__Please choose task 2a or 2b (1pt)__ depending on your model (you can do both, but you will be awarded points for one of these two tasks).

__Task 2a:__ zero-shot translation. Take the first verse of [Edgar Allan Poe's "Raven"](https://www.poetryfoundation.org/poems/48860/the-raven) and __translate it into French.__ (You are free to use any other text of at least the same size)

Original text:

```
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Verify your translation by converting french back into english using a public machine translation service.

__Task 2b:__ toxicity classification for [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations).

Make the model solve binary classification (toxic vs not toxic) in the few shot mode. For few-shot examples, use 2-3 toxic and 2-3 non-toxic non-toxic examples. Measure accuracy on at least 25 samples. You may need to try several different prompts before you find the one that works.

In [ ]:
from datasets import load_dataset
from datasets import concatenate_datasets

ds_test = load_dataset("SetFit/toxic_conversations", split="test")
ds_test = ds_test.shuffle(seed=100)

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
n_positives = 12
n_negatives = 13

positives = ds_test.filter(lambda x: x['label'] == 1).select(range(n_positives))
negatives = ds_test.filter(lambda x: x['label'] == 0).select(range(n_negatives))
test_set = concatenate_datasets([positives, negatives]).shuffle(seed=100)

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Task 2b
from sklearn.metrics import accuracy_score

classification_prompt = """
ROLE
You are an experienced moderator of toxic content.

TASK
You must classify input text as toxic or non-toxic.

Toxic text examples:
1. Why? because we are still figuring out a way to pay for the freaking choo-choo to nowhere.
2. But he is, Blanche! He is a racist!
3. What is most important is that he be independent and loyal nott o Trump or any president but loyal to the Constitution.  Trump is a disgrace and a disaster.  He needs to go back to his phony TV show.

Non-toxic examples:
1. theres not enough going on around here for air service none want to waste there time on this town
2. How about ZERO
3. Her name is still floating? Might I suggest flushing twice?

OUTPUT FORMAT
Write ONLY word 'toxic' if input text is toxic, else write ONLY word 'non-toxic'.

INPUT TEXT: {}
"""

y_true, y_pred = [], []
for item in tqdm(test_set):
  pred_label = generate(classification_prompt.format(item['text']), max_new_tokens=5, do_sample=False)
  y_true.append(item['label'])
  y_pred.append(1 if pred_label == 'toxic' else 0)

print(f'accuracy = {accuracy_score(y_true, y_pred)}')

  0%|          | 0/25 [00:00<?, ?it/s]

accuracy = 0.84


**Описание решения:** я написал промт с указанием роли и задачи для модели, а также со строгими указаниями, в каком формате нужно выводить результат (возвращать только вердикт классификации, без каких-либо пояснений). Я использовал few shot-классификацию, добавив в промт по 3 примера токсичных и нетоксичных текстов из *обучающей* выборки датасета (train split). В параметрах генерации я указал greedy-режим (do_sample=False), потому что в этой задаче от модели требуется не творческая генерация нового контента, а максимально точное следование инструкции.

Для оценки качества решения задачи классификации я выбрал 25 примеров из тестовой части датасета (test split): 12 позитивных (токсичных) и 13 негативных (нетоксичных). При первом же запуске я получил достаточно хорошее значение accuracy = 0.84.


__Task 3 (1pt):__ create a prompt and few-shot examples tha make the model __change the gender pronouns__ of the main actor in a given sentence in any direction of your choice. E.g. the doctor took off _his_ mask <-> the doctor took of _her_ mask.


In [ ]:
pronouns_prompt = """
Change gender pronouns in the input text from male to female.
Examples:
1. the doctor took off his mask -> the doctor took off her mask.
2. The president delivered his inaugural address. -> The president delivered her inaugural address.
3. He looked at himself in the mirror, admiring his new haircut. ->  She looked at herself in the mirror, admiring her new haircut.

OUTPUT FORMAT
You must write only resulting sentence without any explanations.

INPUT TEXT: {}
"""

input_texts = [
    "Noting his emotions in a diary helped him understand himself better.",
    "After the accident, the driver inspected his crashed sedan for damages.",
    "His tone of voice changed once he heard the good news.",
    "As he spoke, a tear gradually rolled down his face.",
    " He trained for years to become a pilot, yet fate decided otherwise."
]

for text in input_texts:
  print(generate(pronouns_prompt.format(text), max_new_tokens=50, do_sample=False))

Noting her emotions in a diary helped her understand herself better.
After the accident, the driver inspected her crashed sedan for damages.
His tone of voice changed once she heard the good news.
As she spoke, a tear gradually rolled down her face.
She trained for years to become a pilot, yet fate decided otherwise.


**Описание решения:** для этой задачи я использовал простой промт с кратким описанием задачи, тремя примерами предложений с изменением гендерных местоимений и описанием формата выходного текста. Для генерации я выбрал режим greedy, т.к. в этой задаче креативность тоже не нужна.

Для тестирования было использовано 5 примеров с сайте Reverso Context. В одном предложении модель ошиблась, не заменив местоимение his на her, но все остальные были изменены верно.

__Task 4 (1pt):__ write a prompt and supply examples such that the model would __convert imperial units to metric units__ (miles -> kilometers; mph -> kph). More specifically, the model should rewrite a given sentence and replace all imperial units with their metric equivalents. After it works with basic distances and speed, try to find complicated examples where it does *not* work.

Please note that 1 mile is not equal to 1 km :)

In [ ]:
units_prompt = """
You must rewrite an input text converting all imperial units in the text (miles, pounds, miles per hour) to metric units (km, kg, kph).

Examples:
1. Running a quarter mile can help improve your speed and stamina. -> Running a 400 m can help improve your speed and stamina.
2. I bought a pound of strawberries to make jam for the winter. -> I bought a 400 g of strawberries to make jam for the winter.
3. On the track, cars can exceed 300 miles per hour. -> On the track, cars can exceed 482 miles per hour.

OUTPUT
write only resulting text with metric units.

INPUT TEXT: {}
"""

input_texts = [
    "The athlete clocked in at 12 miles per hour on the treadmill.",
    "I need a quarter pound of flour to finish this cake.",
    "A cubic yard of wood chips can provide great landscaping mulch.",
    "He felt accomplished after running his very first quarter mile.",
    "They felt worked out after running three miles in the heat."
]

for text in input_texts:
  print(generate(units_prompt.format(text), max_new_tokens=50, do_sample=False))

The athlete clocked in at 19.31 kph on the treadmill.
I need a 113 g of flour to finish this cake.
A cubic yard of wood chips can provide great landscaping mulch.
He felt accomplished after running his very first 400 m.
They felt worked out after running 4.8 km in the heat.


**Описание решения:** я написал промт с заданием переписать текст с заменой всех имперских единиц измерения в тексте на метрические и с указанием строгого формата вывода. Для тестирования были взять 5 предложений с сайта Reverso Context. В итоге модель правильно преобразовала единицы измерения, которые были указаны в примерах в промте, но при этом не преобразовала ярды, которых не было в примерах.

### Part 2: local inference & text-generation internals

In Part 1 we used `model.generate` as a black box. Now let's look **under the hood**: how a language model turns logits into text and implement a decoding strategy by hand.

We already loaded [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base) above: a dense 4B model that fits a free Colab T4 in fp16 (~8 GB).

## Text generation

**Comparison of strategies for language model text generation:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br> **Cons:** Computationally more complex and requires a good loss function. |

Documentation references:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

### Generation with HuggingFace

In [ ]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# greedy inference:                                        do_sample=False)
# beam search for highest probability:                     num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

Input batch (encoded): {'input_ids': tensor([[  785,  1156, 11105, 48195,  1103,  2272,   627,  5868,  1075]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Output: The first discovered martian lifeform looks like a miniature version of an Earth bacterium.

The first discovered martian lifeform looks like a miniature version of an Earth bacterium.

Last Updated on

The first discovered martian lifeform looks like a miniature version of an Earth bacterium.

A team of researchers from the Georgia Institute of Technology and the Florida Institute of


#### Low-level code for text generation

In [ ]:
prompt = "Moscow is the capital of"
# prompt = "Skippy, a young android, likes to dream about electric"

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs.flatten(), num_samples=1)

    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)

Moscow is the capital of 

Step #0 candidates:
ĠRussia   : 0.4651 
Ġwhich    : 0.1559 
Ġthe      : 0.1487 
Ġwhat     : 0.0490 
ĠRussian  : 0.0224 

Chosen token: 



Step #1 candidates:
A         : 0.1619 
Russia    : 0.1544 
Mos       : 0.1451 
What      : 0.1203 
The       : 0.0486 

Chosen token: Mos

Step #2 candidates:
cow       : 0.9995 
k         : 0.0003 
quito     : 0.0000 
<|endoftext|>: 0.0000 
kov       : 0.0000 

Chosen token: cow

Step #3 candidates:
Ġis       : 0.9336 
,         : 0.0282 
<|endoftext|>: 0.0121 
ĊĊ        : 0.0082 
Ġ(        : 0.0045 

Chosen token:  is

Step #4 candidates:
Ġthe      : 0.9917 
Ġa        : 0.0029 
Ġindeed   : 0.0023 
Ġnot      : 0.0008 
Ġactually : 0.0002 

Chosen token:  the

Step #5 candidates:
Ġcapital  : 0.9976 
Ġlargest  : 0.0006 
Ġ**       : 0.0003 
Ġfederal  : 0.0002 
Ġcap      : 0.0001 

Chosen token:  capital

Step #6 candidates:
Ġof       : 0.8711 
Ġcity     : 0.0651 
Ġand      : 0.0631 
Ġ(        : 0.0001 
Ġ**       : 0.0001 

C

**Task 5: write code for nucleus sampling generation (2 points)**:

Use the `nucleus_sampling()` template below. Look at the detailed generation code above for inspiration. __Please do not use model.generate__.

**Bonus task: write code for beam search (3 bonus points)**

In [ ]:
from typing import Tuple, List
@torch.inference_mode()
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    """generates the next token from the nucleus of tokens with cumulative probability up to param:prob"""

    if not (0.0 < prob <= 1.0):
        raise ValueError("prob must be in (0, 1]")

    inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    condition = cumulative_probs >= prob
    if condition.any():
        nucleus_size = int(condition.nonzero(as_tuple=False)[0].item()) + 1
    else:
        nucleus_size = sorted_probs.numel()

    nucleus_probs = sorted_probs[:nucleus_size]
    nucleus_indices = sorted_indices[:nucleus_size]
    nucleus_probs = nucleus_probs / nucleus_probs.sum()
    sampled_idx = torch.multinomial(nucleus_probs, num_samples=1).item()
    sampled_token_id = int(nucleus_indices[sampled_idx].item())
    sampled_token = tokenizer.decode([sampled_token_id])

    possible_tokens = [
        tokenizer.decode([int(token_id)])
        for token_id in nucleus_indices.tolist()
    ]

    # sampled_token should be a string token that was generated
    # possible_tokens should be a list of all tokens that have non-zero probability
    return sampled_token, possible_tokens

In [ ]:
# Tests for nucleus sampling.
prompt = "Elbrus is the highest"

sampled, nucleus = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
print("sampled:", repr(sampled), "| nucleus size:", len(nucleus))

# 1) the sampled token must come from inside the nucleus
assert sampled in nucleus
assert len(nucleus) >= 1

# 2) the nucleus depends only on `prob`, not on the random draw -> it is deterministic across calls
_, nucleus_again = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
assert sorted(nucleus) == sorted(nucleus_again), "the nucleus itself should not be random"

# 3) a larger probability threshold never yields a smaller nucleus
_, small = nucleus_sampling(model, tokenizer, prompt, prob=0.1)
_, large = nucleus_sampling(model, tokenizer, prompt, prob=0.95)
assert len(large) >= len(small), "higher p should include at least as many tokens"

# 4) a tiny threshold collapses to the single most likely token
_, top1 = nucleus_sampling(model, tokenizer, prompt, prob=1e-6)
assert len(top1) == 1, "with p -> 0 the nucleus is just the top-1 token"

print("All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)")

sampled: ' mountain' | nucleus size: 3
All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)


### Part 3: Chain-of-thought prompting (4 points total)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---



In [ ]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

--2026-11-05 16:12:52--  https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130192 (127K) [text/plain]
Saving to: ‘aqua.json’

aqua.json           100%[===================>] 127.14K  --.-KB/s    in 0.003s  

2026-11-05 16:12:52 (42.7 MB/s) - ‘aqua.json’ saved [130192/130192]



In [ ]:
print("Example:")
data[150]

Example:


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?',
 'options': ['A)1 minute',
  'B)2 minutes',
  'C)3 minutes',
  'D)4 minutes',
  'E)5 minutes'],
 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B",
 'correct': 'B'}

### Naive solution

Here, we prompt the model to choose an answer to the example above (`data[150]`) out of the options given above. We're using a format that mimics grade school solution textbook.

Please note that there are minor formatting changes in options: an extra space and an opening bracket. Those may or may not be important :)

In [ ]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [ ]:
# solving an equation directly
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:]  D
Answer: Janice bikes at 1 mile per 6 minutes, while Jennie bikes at 1 per 3 minutes. So together they cover 1/6 + 1/3 = 1/2 miles every three minutes. That means it will take them six minutes to cover one mile.<|endoftext|>


And here's how you can solve this with few-shot chain-of-thought prompting.

You need to chang 3 things
- use a new field called **Rationale**, that contains a step-by-step solution to the problem
- add several few-shot examples of previously solved problems **with rationales**
- change the final prompt so that the model has to generate rationale before answering

In [ ]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [ ]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

__Task 6 (1 pt)__ write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [ ]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
    """

    prompt = ""
    for example in fewshot_examples:
      prompt += f"{QUESTION_PREFIX}{example['question'].strip()}\n{OPTIONS_PREFIX}"
      for i in range(len(example['options'])):
        option = example['options'][i].strip()
        prompt += f'({option[:2]} {option[2:]}'
        if i < len(example['options']) - 1:
          prompt += ' '
      prompt += f'\n{CHAIN_OF_THOUGHT_PREFIX}{example['rationale'].strip()}'
      prompt += f'\n{ANSWER_PREFIX}{example['correct'].strip()}'
      prompt += FEWSHOT_SEPARATOR

    prompt += f"{QUESTION_PREFIX}{main_question['question'].strip()}\n{OPTIONS_PREFIX}"
    for i in range(len(main_question['options'])):
      option = main_question['options'][i].strip()
      prompt += f'({option[:2]} {option[2:]}'
      if i < len(main_question['options']) - 1:
        prompt += ' '
    prompt += f'\n{CHAIN_OF_THOUGHT_PREFIX.strip()}'

    return prompt



generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# Hint: if two prompts do not match, you may find it usefull to use https://www.diffchecker.com or similar to find the difference

Well done!


__Task 7 (1 points):__ Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy.
For each question, peak $n=5$ other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to __stop generating as soon as the model generates Final Answer: [A-E]__
To do so, you can either generate manually (see low-level generation above) or use [transformers stopping criteria](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2), whichever you prefer.

If you do everything right, the model should be much better than random. However, please __do not expect miracles__: this is far from the best models, and it will perform much worse than an average human.

In [ ]:
NUM_SAMPLES = 0    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [ ]:
import numpy as np
np.random.seed(100)

np.random.shuffle(data)
train_set, test_set = data[:len(data) // 2], data[len(data) // 2:]
print(f'train questions: {len(train_set)}')
print(f'test set: {len(test_set)}')

train questions: 127
test set: 127


In [ ]:
answers = []
for question in tqdm(test_set):
  prompt = make_prompt(main_question=question, fewshot_examples=np.random.choice(train_set, size=5))
  output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
  answer = tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu())
  answers.append(answer)

# Optionally, consider inferencing multiple sentences in a batch for faster inference;
# If you choose to batch outputs, make sure the results are the same as with batch=1 (using greedy inference)

train questions: 127
test set: 127


  0%|          | 0/127 [00:00<?, ?it/s]

In [ ]:
def parse_answer(text):
  text = text[text.index('Correct Answer: ') + len('Correct Answer: '):]
  if '<|endoftext|>' in text:
    return text[:text.index('<|endoftext|>')].strip()
  elif 'Question' in text:
    return text[:text.index('Question')].strip()
  return text.strip()

In [ ]:
with open('answers.txt') as f:
  answers = f.read().split('\n===\n')

In [ ]:
for question, answer in zip(test_set, answers):
  NUM_SAMPLES += 1
  if 'Correct Answer: ' in answer:
    NUM_RESPONDED += 1
    model_answer = parse_answer(answer)
    # print(question['correct'], model_answer)
    if question['correct'] == model_answer:
      NUM_CORRECT += 1

In [ ]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.8818897637795275
Accuracy (when responded): 0.2857142857142857
Accuracy (overall): 0.25196850393700787
Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.
Make sure you generate enough tokens that the model can produce a correct answer.
When in doubt, take a look at the full model output. You can often spot errors there.


__Task 8 (2 points)__ Experiment time!

Your final quest is to use the testbench you've just written to answer one of the following questions:

### Option 1: How many shots do you need?

How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale __without__ any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

### Option 2: Is this prompting tecnique reliable?

_Inspired by research by Anton Voronov, Lena Volf and Max Ryabinin._

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?



### Option 3: Inference Matters

There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).


The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique).
Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run `model.generate` on a list with multiple copies of the same prompt.




================================================

__Common rules:__ You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :)
Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)



In [ ]:
# Option 2b

QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
    """

    prompt = ""
    for example in fewshot_examples:
      prompt += f"{QUESTION_PREFIX}{example['question'].strip()}\n{OPTIONS_PREFIX}"
      for i in range(len(example['options'])):
        option = example['options'][i].strip()
        prompt += f'({option[:2]} {option[2:]}'
        if i < len(example['options']) - 1:
          prompt += ' '
      prompt += f'\n{CHAIN_OF_THOUGHT_PREFIX}{example['rationale'].strip()}'
      prompt += f'\n{ANSWER_PREFIX}{example['correct'].strip()}'
      prompt += FEWSHOT_SEPARATOR

    prompt += f"{QUESTION_PREFIX}{main_question['question'].strip()}\n{OPTIONS_PREFIX}"
    for i in range(len(main_question['options'])):
      option = main_question['options'][i].strip()
      prompt += f'({option[:2]} {option[2:]}'
      if i < len(main_question['options']) - 1:
        prompt += ' '
    prompt += f'\n{CHAIN_OF_THOUGHT_PREFIX.strip()}'

    return prompt

In [ ]:
answers = []
for question in tqdm(test_set):
  prompt = make_prompt(main_question=question, fewshot_examples=np.random.choice(train_set, size=5))
  output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
  answer = tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu())
  answers.append(answer)

  0%|          | 0/127 [00:00<?, ?it/s]

In [ ]:
def parse_answer(text):
  text = text[text.index('A: ') + len('A: '):]
  if '<|endoftext|>' in text:
    return text[:text.index('<|endoftext|>')].strip()
  elif 'Q: ' in text:
    return text[:text.index('Q: ')].strip()
  return text.strip()

In [ ]:
NUM_SAMPLES = 0    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [ ]:
for question, answer in zip(test_set, answers):
  NUM_SAMPLES += 1
  if 'A: ' in answer:
    NUM_RESPONDED += 1
    model_answer = parse_answer(answer)
    # print(question['correct'], model_answer)
    if question['correct'] == model_answer:
      NUM_CORRECT += 1

In [ ]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.5275590551181102
Accuracy (when responded): 0.2626865671641791
Accuracy (overall): 0.13858267716535433
Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.
Make sure you generate enough tokens that the model can produce a correct answer.
When in doubt, take a look at the full model output. You can often spot errors there.


После замены слов Question и Answer на Q и A и замены разделителя между примерами с \n\n\n на \n-------\n точность сильно упала: с 25% до 14%.

In [ ]:
# Option 2a

QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_wrong_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
    """

    prompt = ""
    for example in fewshot_examples:
      prompt += f"{QUESTION_PREFIX}{example['question'].strip()}\n{OPTIONS_PREFIX}"
      for i in range(len(example['options'])):
        option = example['options'][i].strip()
        prompt += f'({option[:2]} {option[2:]}'
        if i < len(example['options']) - 1:
          prompt += ' '
      # print(example['rationale'])
      if 'answer' in example['rationale'].lower():
        answer_index = example['rationale'].lower().index('answer')
        rationale = example['rationale'][:answer_index]
      else:
        rationale = example['rationale']
      prompt += f'\n{CHAIN_OF_THOUGHT_PREFIX}{rationale.strip()}'
      options = list(map(lambda s: s[:2], example['options']))
      options = list(filter(lambda s: s != example['correct'], options))
      wrong_answer = random.choice(options)
      prompt += f'\n{ANSWER_PREFIX}{wrong_answer.strip()}'
      prompt += FEWSHOT_SEPARATOR

    prompt += f"{QUESTION_PREFIX}{main_question['question'].strip()}\n{OPTIONS_PREFIX}"
    for i in range(len(main_question['options'])):
      option = main_question['options'][i].strip()
      prompt += f'({option[:2]} {option[2:]}'
      if i < len(main_question['options']) - 1:
        prompt += ' '
    prompt += f'\n{CHAIN_OF_THOUGHT_PREFIX.strip()}'

    return prompt

In [ ]:
NUM_SAMPLES = 0    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [ ]:
answers = []
for question in tqdm(test_set):
  prompt = make_wrong_prompt(main_question=question, fewshot_examples=np.random.choice(train_set, size=5))
  output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
  answer = tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu())
  answers.append(answer)

  0%|          | 0/127 [00:00<?, ?it/s]

In [ ]:
for question, answer in zip(test_set, answers):
  NUM_SAMPLES += 1
  if 'Correct answer: ' in answer:
    NUM_RESPONDED += 1
    model_answer = parse_answer(answer)
    # print(question['correct'], model_answer)
    if question['correct'] == model_answer:
      NUM_CORRECT += 1

In [ ]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.0


ZeroDivisionError: division by zero